In [ ]:
from openai import OpenAI
import json
import os

# Set OpenAI API key
client = OpenAI(
    api_key="",  # This is the default and can be omitted
)

In [ ]:
# Load the JSON file
input_file = "CV_images_tinyllava-6-24-24-test.json"
output_file = "processed_CV_images_tinyllava-6-24-24-test.json"

NUM_ENTRIES_TO_BE_PROCESSED = 10
QUESTIONS_LIST = [
    "Q1: What imaging modality is represented in this image?",
    "Q2: What body region or anatomical area does this image depict? (For microscopic images and other images where the body region cannot be determined, the response could be: Image <modality type>; unable to determine body region based solely on the image.)",
    "Q3: Are there any abnormalities identified in this image?",
    "Q4: Does this image appear normal, or does it show any irregularities?",
    "Q5: Does this image contain any label or index that is significant or noteworthy?",
]

with open(input_file, "r") as file:
    data = json.load(file)
    print(f"Number of entries: {len(data)}")

# Function to process captions and generate answers
def process_caption(caption):

    question_str = ""
    for q in QUESTIONS_LIST:
        question_str += f"{q}\n"

    messages = [
        {"role": "system", "content": "You are a medical expert trained to interpret medical image captions."},
        {"role": "user", "content": f"""
            For the provided caption, answer the following questions strictly based on the caption:

            Caption: {caption}

            {question_str}

            All answers must be derived strictly from the caption and should be based solely on the information provided within the caption. 

            Please provide concise answers for each question.
        """}
    ]
    
    # Call OpenAI API
    # response = openai.ChatCompletion.create(
    #     model="gpt-4",
    #     messages=messages,
    #     max_tokens=300
    # )
    response = client.chat.completions.create(
        model="gpt-4",
        messages=messages,
        response_format={"type": "text"},
        temperature=1,
        max_tokens=2048,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )
    return response

# Process each entry in the data
output = []
for i in range(NUM_ENTRIES_TO_BE_PROCESSED):
    print(f"processing {i} entry")
    entry = data[i]
    gpt_response = entry["conversations"][-1]["value"].strip()  # Get the GPT caption from conversations
    answers_gpt = process_caption(gpt_response).to_dict()  # Process caption with OpenAI API
    answers_list = answers_gpt["choices"][0]["message"]["content"].strip().split("\n")

    conversation_list = []
    for q, a in zip(QUESTIONS_LIST, answers_list):
        conversation_list.append(q)
        conversation_list.append(a)

    answers = entry
    answers["caption"] = gpt_response
    answers["conversations"] = conversation_list
    output.append(answers)

# Save processed data to a new JSON file
with open(output_file, "w") as outfile:
    json.dump(output, outfile, indent=4)

print(f"Processing complete. Answers saved to '{output_file}'.")


Number of entries: 16082
processing 0 entry
processing 1 entry
processing 2 entry
processing 3 entry
processing 4 entry
processing 5 entry
processing 6 entry
processing 7 entry
processing 8 entry
processing 9 entry
Processing complete. Answers saved to 'processed_CV_images_tinyllava-6-24-24-test.json'.
